# Cross-model comparison and analysis

This notebook is the main analysis for the report. It loads predictions from all four models and produces:

1. **Master comparison table** — accuracy + macro-F1 for each model, overall and by agreement tier.
2. **Confusion matrices** for each model.
3. **Agreement-tier breakdown** chart that answers RQ2.
4. **Error analysis** — example misclassifications by the best model.

Prerequisites: every teammate has committed their `*_predictions.csv` to `predictions/`. Expected files:
- `predictions/tfidf_logreg_predictions.csv`     (Gavin)
- `predictions/tfidf_logreg_lm_predictions.csv`  (Noah)
- `predictions/bert_finetuned_predictions.csv`   (Avi)
- `predictions/finbert_finetuned_predictions.csv` (Max)

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    if not os.path.exists('financial-sentiment-comparison'):
        !git clone https://github.com/maximusrome/financial-sentiment-comparison.git
    %cd financial-sentiment-comparison
    !pip install -q pandas scikit-learn pyarrow matplotlib seaborn

from pathlib import Path
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

## Load all predictions

In [ ]:
from evaluation import (
    load_all_predictions, build_comparison_table, pretty_comparison_table,
    evaluate_predictions, plot_confusion_matrix,
    plot_agreement_tier_comparison, find_misclassifications, TIER_ORDER,
)

preds = load_all_predictions('predictions')
print(f'Found predictions for {len(preds)} models:')
for name, df in preds.items():
    print(f'  {name:25s}  n={len(df)}')

## Master comparison table

This is the main results table for the report.

In [ ]:
table = build_comparison_table(preds)
table.to_csv('results/tables/comparison_table.csv', index=False)
table.round(4)

In [ ]:
# Pivoted 'pretty' view for easy reading
wide = pretty_comparison_table(table)
wide

## Confusion matrices (one per model, overall)

In [ ]:
import matplotlib.pyplot as plt

n_models = len(preds)
fig, axes = plt.subplots(1, n_models, figsize=(4 * n_models, 3.6))
if n_models == 1:
    axes = [axes]

for ax, (name, _) in zip(axes, preds.items()):
    res = evaluate_predictions(f'predictions/{name}_predictions.csv')
    plot_confusion_matrix(res['overall']['confusion_matrix'],
                          title=name, ax=ax)

plt.tight_layout()
plt.savefig('results/figures/all_confusion_matrices.png', dpi=150,
            bbox_inches='tight')
plt.show()

## Agreement-tier breakdown (RQ2)

In [ ]:
ax = plot_agreement_tier_comparison(table, metric='macro_f1')
plt.tight_layout()
plt.savefig('results/figures/agreement_tier_f1.png', dpi=150,
            bbox_inches='tight')
plt.show()

In [ ]:
ax = plot_agreement_tier_comparison(table, metric='accuracy')
plt.tight_layout()
plt.savefig('results/figures/agreement_tier_accuracy.png', dpi=150,
            bbox_inches='tight')
plt.show()

## Model-vs-model agreement

Useful sanity check: if BERT and FinBERT agree on 95% of predictions, the "domain-specific pretraining" story is weaker.

In [ ]:
import pandas as pd
import numpy as np

# Build a matrix of predictions
merged = None
for name, df in preds.items():
    d = df[['sentence_id', 'predicted_label']].rename(
        columns={'predicted_label': name})
    merged = d if merged is None else merged.merge(d, on='sentence_id')

names = list(preds.keys())
n = len(names)
mat = np.zeros((n, n))
for i, a in enumerate(names):
    for j, b in enumerate(names):
        mat[i, j] = (merged[a] == merged[b]).mean()

agreement_df = pd.DataFrame(mat, index=names, columns=names).round(3)
agreement_df

## Qualitative error analysis — FinBERT

In [ ]:
errors = find_misclassifications(
    'predictions/finbert_finetuned_predictions.csv', n_examples=8)
cols = ['sentence_id', 'agreement_tier', 'label_name', 'predicted_label', 'text']
errors[cols]

## Key numbers for the report

Run this cell last to produce a cleanly formatted summary you can paste into the report or slides.

In [ ]:
print('=== Summary for report ===\n')
overall = table[table['tier'] == 'overall'].set_index('model')
for model, row in overall.iterrows():
    print(f'{model:25s}  acc={row["accuracy"]:.4f}  '
          f'macro_f1={row["macro_f1"]:.4f}')

print('\n=== By agreement tier (macro-F1) ===\n')
pivot = table.pivot(index='model', columns='tier', values='macro_f1').round(4)
print(pivot.to_string())